### Initialization and Environment Setup

This cell initializes the AutoREACTER environment and prepares the working directory for the current run.

- `Initialization()` sets up the AutoREACTER runtime environment.
- `InputParser()` prepares the parser for validating and normalizing user input files.
- `GetCacheDir()` retrieves the staging cache directory used for intermediate data.
- `RunDirectoryManager.make_dated_run_dir()` creates a timestamped run directory to store outputs for this session.

In [ ]:
from AutoREACTER.initialization import Initialization
from AutoREACTER.input_parser import InputParser
from AutoREACTER.cache import GetCacheDir, RunDirectoryManager, RetentionCleanup
from AutoREACTER.detectors.functional_groups_detector import FunctionalGroupsDetector
from AutoREACTER.detectors.reaction_detector import ReactionDetector
from AutoREACTER.detectors.non_monomer_detector import NonReactantsDetector
from AutoREACTER.reaction_preparation.reaction_processor.prepare_reactions import PrepareReactions
from AutoREACTER.reaction_preparation.lunar_client.molecule_3d_preparation import Molecule3DPreparation
from AutoREACTER.reaction_preparation.lunar_client.lunar_api_wrapper import LunarAPIWrapper
from rdkit import Chem
from rdkit.Chem import Draw
import json

Initialization()
input_parser = InputParser()
cache_dir = GetCacheDir().staging_dir
dated_cache_dir = RunDirectoryManager.make_dated_run_dir(cache_dir, chdir_to="none")
# #future use
# RunDirectoryManager.copy_into_run(cache_dir, dated_cache_dir)
print(f"Cache directory: {cache_dir}")




### Cache Cleanup (Optional)

This cell can be used to clean up cached data generated by previous runs.


In [ ]:

# `RetentionCleanup.run()` removes cached directories from the AutoREACTER cache base directory.
#  Useful for freeing disk space or resetting the cache state.

# RetentionCleanup.run(GetCacheDir().cache_base_dir)

### Load and Validate Monomers

This cell loads the example input file, validates it using the `InputParser`, and visualizes the monomers.



In [ ]:

# validate_inputs() checks the schema and normalizes the data.
# molecule_representation_of_initial_molecules() extracts RDKit molecule objects.
# Draw.MolsToGridImage() displays the monomers in a grid with their labels.

with open("example_1_inputs_count_mode.json", "r") as f:
    input_data = json.load(f)

validated_inputs = input_parser.validate_inputs(input_data)


### Visualize Monomers (Optional)

In [ ]:
initial_molecules = input_parser.initial_molecules_image_grid(validated_inputs)
initial_molecules

### Cache Directory Information

This cell prints the default AutoREACTER cache directory.

In [ ]:
default_cache_dir = GetCacheDir().cache_base_dir
print(f"Default cache directory: {default_cache_dir}")

### Functional Group Detection

This cell runs the functional group detection step.


In [ ]:
# FunctionalGroupsDetector() initializes the detector.
# functional_groups_detector() analyzes the validated monomers.
# It identifies functional groups present in each molecule and generates corresponding visualization images.

# Run the detector only after the inputs have been validated, as it consumes validated_inputs.monomers to execute the detection workflow.
functional_groups_detector = FunctionalGroupsDetector()
functional_groups, functional_groups_imgs = \
    functional_groups_detector.functional_groups_detector(
        validated_inputs.monomers
    )

### Functional Group Visualization (Optional)

This cell visualizes the detected functional groups on the molecules.

In [ ]:
# molecules_to_visualization() highlights the functional groups identified in the previous step.
# The resulting image shows each molecule with the detected functional groups marked.

img = functional_groups_detector.functional_group_highlighted_molecules_image_grid(functional_groups_imgs)
img


### Reaction Detection

This cell identifies possible reactions between the detected functional groups.


In [ ]:

# ReactionDetector() initializes the reaction detection module.
# reaction_detector() analyzes the detected functional groups and generates possible reaction instances.
# create_reaction_image_grid() visualizes the detected reactions in a grid format.

reaction_detector = ReactionDetector()
reaction_instances = reaction_detector.reaction_detector(functional_groups)


### Reaction visualization (Optional)

In [ ]:
img = reaction_detector.available_reaction_image_grid(reaction_instances)
img

### Reaction Selection

This cell filters and selects the relevant reactions from the detected reaction instances.


In [ ]:
# reaction_selection() processes the detected reactions.
# It removes user selected reactions from the list of detected reactions and returns the final set of reactions to be used for template generation.

selected_reactions = reaction_detector.reaction_selection(reaction_instances)

In [ ]:
print(selected_reactions)

### Non-Reactant (Non-Monomer) Detection

This cell identifies molecules that do **not participate in any detected reactions**.



In [ ]:
# NonReactantsDetector() initializes the detector.
# non_monomer_detector() to determine which molecules are non-reactive.


non_monomer_detector = NonReactantsDetector()
non_reactants_list = non_monomer_detector.non_monomer_detector(validated_inputs, selected_reactions)


### non_reactants_to_visualization

In [ ]:
# `()` grid to non reactive molecules

# If no non-reactant molecules are detected, this stage is **automatically skipped**, and the workflow continues without generating any visualization.

img_non_reactants = non_monomer_detector.non_reactants_to_visualization(non_reactants_list)
img_non_reactants

### Update Inputs After Non-Reactant Filtering

This cell updates the validated inputs after identifying non-reactive molecules.


In [ ]:
# non_reactant_selection() removes or marks molecules that do not participate in any reactions.
# The resulting `updated_inputs` contains the filtered set of monomers that will be used in the next stages of the workflow.
# If no non-reactants were detected in the previous step, the inputs remain unchanged.

updated_inputs = non_monomer_detector.non_reactant_selection(validated_inputs, non_reactants_list)

In [ ]:
prepare_reactions = PrepareReactions(cache_dir)
prepared_reactions = prepare_reactions.prepare_reactions(selected_reactions)

In [ ]:
# You can select different highlight types to visualize different aspects of the reactions, by highlight tag
# such as "edge" for egde atoms of the templates,
# "delete" for removed components (optional based on the reaction), 
# or "initiators" for reaction initiator atoms. 
# By default, it highlights all changes in the reaction templates.
img = prepare_reactions.reaction_templates_highlighted_image_grid(prepared_reactions, "edge")
img

In [ ]:

molecule3dpreparation = Molecule3DPreparation(cache_dir)
updated_inputs_with_3d_mols, prepared_reactions_with_3d_mols = molecule3dpreparation.prepare_molecule_3d_geometry(updated_inputs, prepared_reactions)

        

In [ ]:

lunar_api_wrapper = LunarAPIWrapper(cache_dir)
lunar_results = lunar_api_wrapper.lunar_workflow(updated_inputs_with_3d_mols, prepared_reactions_with_3d_mols)

In [ ]:
print(lunar_results)

In [ ]:
from AutoREACTER.reaction_preparation.lunar_client.REACTER_files_builder import REACTERFilesBuilder
REACTER_files_builder = REACTERFilesBuilder(cache_dir=cache_dir, updated_inputs_with_3d_mols=updated_inputs_with_3d_mols)
reacter_files = REACTER_files_builder.molecule_template_preparation(lunar_files=lunar_results,
    prepared_reactions_with_3d_mols=prepared_reactions_with_3d_mols
)

In [ ]:
print(reacter_files)